# ♟️ Chess Numba — Google Colab 雲端搜尋調參執行器 (LMR Phase 2)

### 🏆 Phase 1 成果總結
* **Phase 1 冠軍基準**：六軸基礎 LMR 參數已全數定案寫入生產基準（`BASE 437 / HIST 204 / CUTNODE 1839 / TTCAP 1191 / MC 35 / TABLE 113%`），L2 實測 **+15.6 ± 16.9 Elo**！

### 🎯 Phase 2 核心調參六軸（戰術吃子、殺著反制與局勢動態）
1. `LMR_BAD_CAPTURE_BONUS`（當前基線 1046）：劣勢吃子（SEE < 0）額外減步懲罰。
2. `LMR_GOOD_CAPTURE_RELIEF`（當前基線 1030）：優勢吃子（SEE >= 0）減步寬減保護。
3. `LMR_KILLER_COUNTER_RELIEF`（當前基線 1024）：殺著步與對策步（Killer / Countermove）寬減保護。
4. `LMR_NO_TTMOVE_BONUS`（當前基線 1068）：局面無 TT Move 指引時的額外減步懲罰。
5. `LMR_TTMOVE_REDUCTION`（當前基線 1024）：走法命中 TT 走步時的基準減步。
6. `LMR_NOT_IMP_NUM`（當前基線 197）：靜態評估未改善（Not Improving）狀態下的動態減步係數。

### ⚡ 調參架構
* **L1 雲端 SPSA 探索**：80 輪 × 150 局 = 12,000 局 @ 50,000 nodes（inprocess 平行加速）
* **Polyak-Ruppert 平滑取樣**：嚴格鎖定最優經驗比例 **`--tail 0.3`**（後 24 輪 / 3,600 局平滑）
* **L2 深度獨立驗收**：600 局 @ 300,000 nodes，主開局庫 `data/openings.epd`，SPRT(0.0, 3.0)

In [4]:
# 1. 檢視 Colab 雲端硬體規格
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!python3 --version

CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
Python 3.13.16


In [2]:
# 2. 克隆/拉取最新代碼庫並安裝相依套件
import os
branch = "feat-initial-bitboard-setup"

if not os.path.exists("/content/chess_numba"):
    %cd /content
    !git clone -b {branch} https://github.com/b08203031/chess_numba.git

%cd /content/chess_numba
!git checkout -f {branch}
# 全面清除衝突的 json 結果檔，確保 git pull 100% 成功拉取最新 Tail 0.3 基準代碼
!rm -f tournament_analysis/*.json tune_search/candidate*.json
!git pull origin {branch}
!pip install -q numba chess numpy


/content/chess_numba
Already on 'feat-initial-bitboard-setup'
Your branch is up to date with 'origin/feat-initial-bitboard-setup'.
From https://github.com/b08203031/chess_numba
 * branch            feat-initial-bitboard-setup -> FETCH_HEAD
Already up to date.


In [5]:
# 3. 測試環境與驗證 Phase 1 / Phase 2 基準常數
import sys
sys.path.insert(0, "/content/chess_numba")
from tune_search.search_param_registry import get_default_params

phase1_check = ["LMR_BASE_OFFSET", "LMR_TABLE_SCALE_PERCENT", "LMR_CUTNODE_BONUS", "LMR_TTCAPTURE_BONUS"]
phase2_params = [
    "LMR_BAD_CAPTURE_BONUS",
    "LMR_GOOD_CAPTURE_RELIEF",
    "LMR_KILLER_COUNTER_RELIEF",
    "LMR_NO_TTMOVE_BONUS",
    "LMR_TTMOVE_REDUCTION",
    "LMR_NOT_IMP_NUM"
]
print("🔍 檢查 Phase 1 底座（應為 Tail 0.3 冠軍值 437 / 113 / 1839 / 1191）：")
print(get_default_params(names=phase1_check))
print("\n✅ 成功載入！Phase 2 核心六軸當前基準值（Baseline）：")
print(get_default_params(names=phase2_params))


🔍 檢查 Phase 1 底座（應為 Tail 0.3 冠軍值 437 / 113 / 1839 / 1191）：
{'LMR_BASE_OFFSET': 437, 'LMR_TABLE_SCALE_PERCENT': 113, 'LMR_CUTNODE_BONUS': 1839, 'LMR_TTCAPTURE_BONUS': 1191}

✅ 成功載入！Phase 2 核心六軸當前基準值（Baseline）：
{'LMR_BAD_CAPTURE_BONUS': 1046, 'LMR_GOOD_CAPTURE_RELIEF': 1030, 'LMR_KILLER_COUNTER_RELIEF': 1024, 'LMR_NO_TTMOVE_BONUS': 1068, 'LMR_TTMOVE_REDUCTION': 1024, 'LMR_NOT_IMP_NUM': 197}


In [6]:
# 4. 啟動 L1 SPSA LMR 第二階段核心六軸雲端對抗調參（目標 80 輪 x 150 局 = 12,000 局）
# 注意：必須基於最新的 Phase 1 Tail 0.3 冠軍底座開跑！
import os
import json

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"偵測到 Colab CPU 核心數: {cpu_count}，調參並行度設為: {concurrency}")

log_file = "tune_search/logs/lmr_p2_deep_l1.jsonl"
state_file = "tune_search/states/lmr_p2_deep_l1.json"
total_campaign_iters = 80

# 讀取存檔狀態以計算剩餘需要執行的迭代輪數（支援斷點續跑）
if os.path.exists(state_file):
    with open(state_file, "r", encoding="utf-8") as f:
        saved_state = json.load(f)
    done_iters = saved_state.get("iteration", 0)
    remaining_iters = max(0, total_campaign_iters - done_iters)
    resume_flag = "--resume"
    print(f"🔄 斷點續跑模式：已完成 {done_iters}/{total_campaign_iters} 輪，本次接續執行剩餘 {remaining_iters} 輪！")
else:
    done_iters = 0
    remaining_iters = total_campaign_iters
    resume_flag = ""
    print(f"🚀 全新開跑模式：計劃執行全量 {remaining_iters} 輪！")

if remaining_iters > 0:
    !python -m tune_search.search_tuner \
      --params LMR_BAD_CAPTURE_BONUS,LMR_GOOD_CAPTURE_RELIEF,LMR_KILLER_COUNTER_RELIEF,LMR_NO_TTMOVE_BONUS,LMR_TTMOVE_REDUCTION,LMR_NOT_IMP_NUM \
      --backend inprocess \
      --scale-mode integer \
      --iter {remaining_iters} \
      --games 150 \
      --nodes 50000 \
      --concurrency {concurrency} \
      --adjudicate \
      --log {log_file} \
      --state {state_file} \
      {resume_flag}
else:
    print("✅ Phase 2 的 80 輪調參已全部完成！請直接執行步驟 5 產出候選參數。\n")


偵測到 Colab CPU 核心數: 8，調參並行度設為: 7
🚀 全新開跑模式：計劃執行全量 80 輪！
[inprocess] openings=500 nodes=50000 concurrency=7 tt=16MB
[pool] allocate worker pair 1/7 (tt=16MB)...
[pool] allocate worker pair 2/7 (tt=16MB)...
[pool] allocate worker pair 3/7 (tt=16MB)...
[pool] allocate worker pair 4/7 (tt=16MB)...
[pool] allocate worker pair 5/7 (tt=16MB)...
[pool] allocate worker pair 6/7 (tt=16MB)...
[pool] allocate worker pair 7/7 (tt=16MB)...
[warmup] JIT nodes=5000 (may take several minutes first time)...
[warmup] done in 1.0s
[warmup] JIT nodes=1500 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 2/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[inprocess] touch worker 3/7
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0.0s
[warmup] JIT nodes=800 (may take several minutes first time)...
[warmup] done in 0

In [3]:
# 5. 分析 SPSA 調參成果，以 Polyak-Ruppert 後 30% 平均（tail 0.3）產出最佳候選參數
!python tune_search/summarize_run.py tune_search/logs/lmr_p2_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_lmr_p2_deep.json
!python tune_search/summarize_run.py tune_search/logs/lmr_p2_deep_l1.jsonl --tail 0.3 --weight-mode uniform --out tune_search/candidate_lmr_p2_tail03_uniform.json
!python tune_search/summarize_run.py tune_search/logs/lmr_p2_deep_l1.jsonl --tail 0.3 --weight-mode linear --out tune_search/candidate_lmr_p2_tail03_linear.json
print("\n=== Phase 2 等權最佳候選參數 JSON (tail 0.3 uniform) ===")
!cat tune_search/candidate_lmr_p2_deep.json
print("\n=== Phase 2 線性加權候選參數 JSON (tail 0.3 linear) ===")
!cat tune_search/candidate_lmr_p2_tail03_linear.json


iterations=80 games=12000 mean_score=0.4964
score sd observed=0.0307 noise-only expected=0.0309 ratio=0.99
  -> ratio ~1: batch scores are indistinguishable from sampling noise; this run carries (almost) no gradient signal. Use more games per batch / more batches before trusting any parameter movement.

param                             default      last  tail_avg moved%rng  drift_z
LMR_BAD_CAPTURE_BONUS                1046    1086.5    1059.5      2.25     0.70  (noise)
LMR_GOOD_CAPTURE_RELIEF              1030    1047.7    1045.8      0.99     0.31  (noise)
LMR_KILLER_COUNTER_RELIEF            1024     926.2     921.5     -5.43    -1.70  (noise)
LMR_NO_TTMOVE_BONUS                  1068    1168.2    1131.7      4.01     0.87  (noise)
LMR_TTMOVE_REDUCTION                 1024     938.1     934.2     -3.43    -0.75  (noise)
LMR_NOT_IMP_NUM                       197     211.3     208.4      3.73     1.00  (noise)

averaged candidate (last 24 iterations):
--param LMR_BAD_CAPTURE_BONUS=10

In [ ]:
# 6. 啟動 L2 深度 SPRT 實戰驗證（300,000 nodes, 600 局，主開局庫獨立驗證）
# 對照基準明確指定為原始 Baseline (tune_search/baseline_lmr_p2.json)
import os

cpu_count = os.cpu_count() or 2
concurrency = max(1, cpu_count - 1 if cpu_count > 2 else cpu_count)
print(f"啟動 L2 驗收，使用並行度: {concurrency}")

!python tools/run_search_param_match.py \
  --params-json tune_search/candidate_lmr_p2_deep.json \
  --baseline-json tune_search/baseline_lmr_p2.json \
  --nodes 300000 \
  --games 600 \
  --openings data/openings.epd \
  --concurrency {concurrency} \
  --sprt 0,3 \
  --batch-games 50 \
  --adjudicate \
  --out tournament_analysis/l2_verify_lmr_p2_deep_tail03.json
